# Credit Default Risk — Probability Review

Run the training pipeline first. These visuals support calibration and portfolio risk-band review; they are not approval rules.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ARTIFACTS = Path("../artifacts")
predictions = pd.read_csv(ARTIFACTS / "test_predictions.csv")
metrics = json.loads((ARTIFACTS / "metrics.json").read_text())
metrics


In [ ]:
predictions["probability_band"] = pd.cut(
    predictions["default_probability"],
    bins=[0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1],
    include_lowest=True,
)
bands = predictions.groupby("probability_band", observed=True).agg(
    Accounts=("actual", "size"),
    ObservedDefaultRate=("actual", "mean"),
    AverageProbability=("default_probability", "mean"),
).reset_index()
figure, axis = plt.subplots(figsize=(11, 5))
axis.plot(range(len(bands)), bands["ObservedDefaultRate"], marker="o", label="Observed")
axis.plot(range(len(bands)), bands["AverageProbability"], marker="o", label="Predicted")
axis.set(title="Observed vs Predicted Default by Probability Band", xlabel="Probability band", ylabel="Rate")
axis.set_xticks(range(len(bands)), bands["probability_band"].astype(str), rotation=45)
axis.legend()
axis.grid(alpha=0.25)
figure.tight_layout()


In [ ]:
figure, axis = plt.subplots(figsize=(9, 5))
for label, group in predictions.groupby("actual"):
    axis.hist(group["default_probability"], bins=20, alpha=0.55, label=f"Actual {label}")
axis.axvline(metrics["decision_threshold"], color="#DC2626", linestyle="--", label="Selected threshold")
axis.set(title="Held-out Default Probability Distribution", xlabel="Probability", ylabel="Accounts")
axis.legend()
figure.tight_layout()
